In [15]:
import pandas as pd
import unicodedata

# ── FUNCION PARA LIMPIAR TEXTO ─────────────────────────
def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto)
    
    # quitar tildes
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    
    # pasar a mayúsculas
    texto = texto.upper()
    
    # quitar espacios dobles
    texto = " ".join(texto.split())
    
    return texto

In [ ]:
# ── RUTAS ──────────────────────────────────────────────
RUTA = "/home/jovyan/work"
ARCHIVO_HISTORICO = RUTA + "/data_historico.csv"
ARCHIVO_PREDICCIONES = RUTA + "/data_predicciones.csv"
ARCHIVO_DATA = RUTA + "/data.csv"

SALIDA = RUTA + "/matriz_riesgo_programas.csv"

In [17]:
# ── CARGA DE DATOS ─────────────────────────────────────
df_hist = pd.read_csv(ARCHIVO_HISTORICO)
df_pred = pd.read_csv(ARCHIVO_PREDICCIONES)
df_data = pd.read_csv(ARCHIVO_DATA)

In [18]:
# ── PROMEDIOS POR PROGRAMA ─────────────────────────────
hist_avg = (
    df_hist
    .groupby("codigo_programa")["matriculados"]
    .mean()
    .reset_index()
    .rename(columns={"matriculados": "promedio_historico"})
)

pred_avg = (
    df_pred
    .groupby("codigo_programa")["matriculados"]
    .mean()
    .reset_index()
    .rename(columns={"matriculados": "promedio_prediccion"})
)

In [19]:
# ── UNIÓN DE DATASETS ──────────────────────────────────
df_merge = pd.merge(hist_avg, pred_avg, on="codigo_programa", how="inner")

# ── CÁLCULO DE VARIACIÓN PORCENTUAL ────────────────────
df_merge["variacion_pct"] = (
    (df_merge["promedio_prediccion"] - df_merge["promedio_historico"]) 
    / df_merge["promedio_historico"]
) * 100

# ── TRAER NOMBRE DEL PROGRAMA ──────────────────────────
df_nombres = (
    df_data[["codigo_programa", "programa"]]
    .drop_duplicates()
)

df_final = pd.merge(df_merge, df_nombres, on="codigo_programa", how="left")

# ── NOMBRES ────────────────────────────────────────────
df_nombres = (
    df_data[["codigo_programa", "programa"]]
    .drop_duplicates()
)

# limpiar nombres
df_nombres["programa"] = df_nombres["programa"].apply(limpiar_texto)

# ── MERGE FINAL ────────────────────────────────────────
df_final = pd.merge(df_merge, df_nombres, on="codigo_programa", how="left")

In [20]:
# ── ORDENAR COLUMNAS ───────────────────────────────────
df_final = df_final[
    [
        "codigo_programa",
        "programa",
        "promedio_historico",
        "promedio_prediccion",
        "variacion_pct"
    ]
]


# ── CLASIFICACIÓN RIESGO ───────────────────────────────────
def clasificar_riesgo(x):
    if x < -20:
        return "Alto"
    elif x < 5:
        return "Medio"
    else:
        return "Bajo"

df_final["nivel_riesgo"] = df_final["variacion_pct"].apply(clasificar_riesgo)

In [21]:
# ── FUNCIÓN DE CATEGORIZACIÓN ──────────────────────────
def clasificar_categoria(nombre):
    if pd.isnull(nombre):
        return "PREGRADOS"
    
    nombre = nombre.upper()
    
    if nombre.startswith("DOCTORADO"):
        return "DOCTORADOS"
    
    elif nombre.startswith((
        "ESPECIALIDAD", "ESPECIALIZACIN", "ESPECIALIZACION",
        "ESPECIALIZACION:", "ESPECIALIZALICION", "ESPECILIZACION"
    )):
        return "ESPECIALIZACIONES"
    
    elif nombre.startswith((
        "MAESTRA", "MAESTRAS", "MAESTRIA", "MAESTRIAS"
    )):
        return "MAESTRIAS"
    
    elif nombre.startswith((
        "TCNICA", "TCNICO", "TECNICA", "TECNICO",
        "TECNOLGIA", "TECNOLOGA", "TECNOLOGIA",
        "TECNOLOGO", "TECNOLLOGIA", "TECNONOLOGIA", "TEGNOLOGIA"
    )):
        return "TECNICOS Y TECNOLOGIAS"
    
    else:
        return "PREGRADOS"

# ── APLICAR CLASIFICACIÓN ──────────────────────────────
df_final["categoria"] = df_final["programa"].apply(clasificar_categoria)

In [ ]:
df_final["programa"] = df_final["programa"].str.strip().str.upper()
df_final["programa"] = df_final["programa"].str.replace(",", " | ", regex=False)

df_agrupado = df_final.groupby("programa").agg({
    "promedio_historico": "mean",
    "promedio_prediccion": "mean",
    "variacion_pct": "mean"
}).reset_index()


df_agrupado = df_final.groupby("programa").agg({
    "promedio_historico": "mean",
    "promedio_prediccion": "mean",
    "variacion_pct": "mean",
    "categoria": "first"  # o "mode" si puede variar
}).reset_index()

def clasificar_riesgo(x):
    if x < -0.2:
        return "Alto"
    elif x < 0.05:
        return "Medio"
    else:
        return "Bajo"

df_agrupado["nivel_riesgo"] = df_agrupado["variacion_pct"].apply(clasificar_riesgo)

In [23]:
# ── EXPORTAR A CSV ─────────────────────────────────────
df_agrupado.to_csv(SALIDA, index=False)

print("Archivo generado en:")
print(SALIDA)

# ── VISTA PREVIA ───────────────────────────────────────
df_final.head()

Archivo generado en:
C:/Users/andpu/OneDrive/Escritorio/ESPECIALIZACION EN ANALITICA DE BIG DATA/SEMESTRE 2/GESTIÓN DE PROYECTOS PARA ANÁLITICA/5_Modulo3_Ejecución_del_Proyecto/SCRIPTS FINAES/DATA/matriz_riesgo_programas.csv


,codigo_programa,programa,promedio_historico,promedio_prediccion,variacion_pct,nivel_riesgo,categoria
0,1,INGENIERIA AGRONOMICA,365.545455,462.15,26.427506,Bajo,PREGRADOS
1,2,MEDICINA VETERINARIA,308.045455,359.25,16.622399,Bajo,PREGRADOS
2,3,ZOOTECNIA,204.954545,257.90,25.832779,Bajo,PREGRADOS
3,4,DISENO GRAFICO,161.022727,194.70,20.914608,Bajo,PREGRADOS
4,4,DISENO GRAFICO,161.022727,194.70,20.914608,Bajo,PREGRADOS
